# Thu thập tài liệu tham khảo

Phụ trách: Trần Nguyên Tân — 24120438.

Chương này trả lời ba câu hỏi về bước lấy trích dẫn (references) của từng bài:

1. Lấy được trích dẫn cho bao nhiêu bài, và lỗi xảy ra ở đâu?
2. Trích dẫn có đủ thông tin để M3 ghép với BibTeX không (ID arXiv, tiêu đề, tác giả, năm)?
3. Gọi Semantic Scholar (S2) tốn bao nhiêu request, và bị giới hạn tốc độ bao nhiêu lần?

Notebook chỉ **đọc** các file mà `main.py` đã ghi trong `data/shards/<MSSV>/`. Notebook không gọi API, nên khi dữ liệu không đổi thì mỗi lần build sách cho cùng một kết quả.

## Mục lục

1. Dữ liệu và cách tính
2. Semantic Scholar và giới hạn tốc độ
3. Dự phòng bằng .bib và .bbl
4. Độ phủ và phân bố trích dẫn
5. Kết quả, nhận xét và giới hạn

In [ ]:
import inspect
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator
import pandas as pd

# Tìm thư mục gốc repo (nơi có pyproject.toml) để notebook chạy được từ docs/ hoặc từ gốc.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.crawler import reference_scraper as rs  # REFERENCE_KEYS: 11 cột của references.parquet

# Thư mục chứa các shard. Mặc định là data/shards của repo.
# Đặt biến môi trường CH03_DATA_ROOT để đọc từ nơi khác, ví dụ khi thử với dữ liệu giả.
SHARDS_DIR = Path(os.environ.get("CH03_DATA_ROOT", ROOT / "data" / "shards"))

MEMBERS = {
    "24120282": "Nguyễn Quốc Đạt",
    "24120237": "Nguyễn Đình Tuấn",
    "24120438": "Trần Nguyên Tân",
    "24120470": "Dương Phan Minh Trí",
    "24120474": "Trịnh Vỹ Triết",
}

# Tên lớp lỗi ghi trong stats.json -> nhóm kết quả dùng trong chương.
STATUS_GROUP = {
    "ok": "ok",
    "PaperNotFoundError": "not_found",
    "ReferencesUnavailableError": "unavailable",
    "ReferenceScrapeError": "error",
}

# Màu theo palette tham chiếu của skill dataviz: slot 1 (xanh) cho một chuỗi dữ liệu.
# Chữ và lưới dùng màu văn bản, không dùng màu dữ liệu.
SURFACE = "#fcfcfb"
BLUE = "#2a78d6"
INK, INK_SECONDARY, INK_MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK_SECONDARY,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.edgecolor": AXIS, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 10, "font.size": 9,
    "figure.dpi": 150,
})

In [ ]:
def load_shard(member_id):
    """Đọc references.parquet và stats.json của một MSSV. Thiếu references.parquet thì trả None.

    Thiếu stats.json, hoặc stats.json không có cờ is_mock, thì coi là DỮ LIỆU MẪU (cùng quy tắc với chương 05).
    """
    folder = SHARDS_DIR / member_id
    refs_path = folder / "references.parquet"
    if not refs_path.exists():
        return None
    stats_path = folder / "stats.json"
    stats = json.loads(stats_path.read_text(encoding="utf-8")) if stats_path.exists() else {}
    refs = pd.read_parquet(refs_path)
    refs.insert(0, "member", member_id)
    return {"refs": refs, "stats": stats, "is_mock": stats.get("is_mock", True)}


shards = {}
for member_id in MEMBERS:
    loaded = load_shard(member_id)
    if loaded is not None:
        shards[member_id] = loaded

IS_MOCK = any(s["is_mock"] for s in shards.values())
# Khi có dữ liệu mẫu, mọi tiêu đề hình có thêm nhãn này để không ai nhầm với số thật.
TAG = "[DỮ LIỆU MẪU] " if IS_MOCK else ""

print(f"Thư mục shard: {SHARDS_DIR}")
print(f"Phạm vi: {len(shards)}/{len(MEMBERS)} thành viên có references.parquet")
if not shards:
    print("CHƯA CÓ DỮ LIỆU THẬT: chưa có references.parquet nào. Các mục phân tích sẽ in thông báo, không có số.")
if IS_MOCK:
    print("CẢNH BÁO: có shard là DỮ LIỆU MẪU. Không dùng các số dưới đây làm kết quả.")
for member_id, s in shards.items():
    status = s["stats"].get("status")
    label = "DỮ LIỆU MẪU" if s["is_mock"] else "số liệu thật"
    print(f"  {member_id} {MEMBERS[member_id]}: {len(s['refs'])} trích dẫn, {label}")
    if status not in (None, "completed"):
        print(f"  ! {member_id}: lần chạy chưa hoàn tất (status = {status})")

In [ ]:
# refs: mỗi dòng là một trích dẫn, kể cả trích dẫn không có ID arXiv.
REF_COLUMNS = ["member", *rs.REFERENCE_KEYS]
if shards:
    refs = pd.concat([s["refs"] for s in shards.values()], ignore_index=True)
    missing = [c for c in rs.REFERENCE_KEYS if c not in refs.columns]
    if missing:
        raise ValueError(f"references.parquet thiếu cột {missing}; kiểm tra lại schema với Trí")
else:
    refs = pd.DataFrame(columns=REF_COLUMNS)

# papers: mỗi dòng là một bài đã thử lấy trích dẫn, đọc từ reference_status trong stats.json.
# Bài không có trích dẫn nào không xuất hiện trong references.parquet, nên phải đọc từ đây.
PAPER_COLUMNS = ["member", "arxiv_id", "status", "n_refs", "n_refs_arxiv", "via"]
paper_frames = []
for member_id, s in shards.items():
    status_by_paper = s["stats"].get("reference_status")
    if status_by_paper is None:
        continue
    per_paper = s["refs"].groupby("arxiv_id").agg(
        n_refs=("ref_arxiv_id", "size"),         # mọi trích dẫn của bài
        n_refs_arxiv=("ref_arxiv_id", "count"),  # chỉ trích dẫn có ID arXiv (count bỏ qua ô rỗng)
        via=("source", "first"),                 # semantic_scholar hoặc bib_fallback
    )
    frame = pd.DataFrame({
        "status": {aid: STATUS_GROUP.get(name, "error") for aid, name in status_by_paper.items()},
    })
    frame = frame.join(per_paper, how="left")
    frame[["n_refs", "n_refs_arxiv"]] = frame[["n_refs", "n_refs_arxiv"]].fillna(0).astype(int)
    frame["via"] = frame["via"].fillna("")
    frame.insert(0, "member", member_id)
    paper_frames.append(frame.rename_axis("arxiv_id").reset_index())
papers = pd.concat(paper_frames, ignore_index=True) if paper_frames else pd.DataFrame(columns=PAPER_COLUMNS)

HAS_REFS = len(refs) > 0
HAS_PAPERS = len(papers) > 0
print(f"{len(refs)} trích dẫn; {len(papers)} bài có trạng thái lấy trích dẫn")
if shards and not HAS_PAPERS:
    print("Chưa có reference_status trong stats.json: các số theo bài sẽ chờ. Cần Triết ghi trường này.")

## 1. Dữ liệu và cách tính

Một bài được đưa vào chương khi `main.py` đã thử lấy trích dẫn của nó. Mẫu số của mọi tỷ lệ là **N = số bài đã thử**, đếm trong `reference_status` của `stats.json`. Không dùng số bài trong khoảng ID làm mẫu số, vì bài chưa được thử thì không có kết quả.

Mỗi bài được đánh giá theo ba tiêu chí, cùng mẫu số N:

- **(a) Lấy không lỗi:** S2 trả lời, có hoặc không có trích dẫn; hoặc bài được dự phòng bằng `.bib` (mục 3).
- **(b) Có ít nhất một trích dẫn.** Bài S2 trả 0 trích dẫn đạt (a) nhưng không đạt (b).
- **(c) Có ít nhất một trích dẫn có ID arXiv.** Chỉ những trích dẫn này vào bảng công bố, và M3 chỉ ghép được với chúng.

Bài lỗi không đạt cả ba tiêu chí. Bốn nhóm trạng thái trong `reference_status`: `ok`; `not_found` (S2 404 và không có .bib/.bbl); `unavailable` (S2 báo có trích dẫn nhưng không đưa mục nào); `error` (mạng lỗi hoặc 429 hết lượt thử).

In [ ]:
if not HAS_PAPERS:
    print("Chưa có dữ liệu: cần reference_status trong stats.json.")
else:
    n = len(papers)
    ok = papers["status"].eq("ok")
    has_ref = papers["n_refs"].gt(0)
    has_arxiv = papers["n_refs_arxiv"].gt(0)
    print(f"N = {n} bài đã thử lấy trích dẫn")
    print(f"(a) lấy không lỗi:                 {ok.sum()}/{n} = {ok.mean():.1%}")
    print(f"(b) có >= 1 trích dẫn:             {has_ref.sum()}/{n} = {has_ref.mean():.1%}")
    print(f"(c) có >= 1 trích dẫn có arXiv ID: {has_arxiv.sum()}/{n} = {has_arxiv.mean():.1%}")

    # Thứ tự cố định, để nhóm không đổi chỗ khi số liệu thay đổi.
    GROUP_ORDER = [
        "S2 trả trích dẫn",
        "S2 trả 0 trích dẫn",
        "Dự phòng từ .bib/.bbl",
        "S2 404, không có .bib/.bbl",
        "S2 có bài nhưng không trả trích dẫn",
        "Lỗi (mạng hoặc 429 hết lượt)",
    ]
    ERROR_GROUP = {
        "not_found": "S2 404, không có .bib/.bbl",
        "unavailable": "S2 có bài nhưng không trả trích dẫn",
        "error": "Lỗi (mạng hoặc 429 hết lượt)",
    }

    def group_of(row):
        """Xếp một bài vào một nhóm để vẽ hình 1."""
        if row["status"] != "ok":
            return ERROR_GROUP[row["status"]]
        if row["via"] == "bib_fallback":
            return "Dự phòng từ .bib/.bbl"
        return "S2 trả trích dẫn" if row["n_refs"] > 0 else "S2 trả 0 trích dẫn"

    counts = papers.apply(group_of, axis=1).value_counts().reindex(GROUP_ORDER, fill_value=0)
    table1 = pd.DataFrame({"số bài": counts, "tỷ lệ (%)": (100 * counts / n).round(1)})
    display(table1)

    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.barh(counts.index, counts.values, color=BLUE, height=0.6)
    ax.invert_yaxis()  # nhóm đầu danh sách nằm trên cùng
    for y, value in enumerate(counts.values):
        ax.text(value + 0.05, y, f"{value} ({value / n:.1%})", va="center", color=INK_SECONDARY)
    ax.set_xlim(0, max(1, counts.max()) * 1.35)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))  # số bài là số nguyên
    ax.set_xlabel(f"Số bài (n = {n})")
    ax.set_title(f"{TAG}Hình 1. Kết quả lấy trích dẫn theo nhóm")
    ax.grid(axis="x")
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    plt.tight_layout()
    plt.show()

    print(f"Trong {n} bài: {ok.sum()} bài lấy được không lỗi, "
          f"trong đó {counts['Dự phòng từ .bib/.bbl']} bài nhờ .bib; "
          f"{counts['Lỗi (mạng hoặc 429 hết lượt)']} bài lỗi.")

## 2. Semantic Scholar và giới hạn tốc độ

Theo đề M1, S2 giới hạn số request. Không có khóa API thì tối đa 1 request mỗi giây và 100 request mỗi 5 phút; có khóa miễn phí thì ổn định ở 1 request mỗi giây. Vượt hạn mức, S2 trả mã **429** (Too Many Requests: quá nhiều yêu cầu).

Code xử lý ba việc để không mất dữ liệu khi gặp 429:

- **Giới hạn tốc độ chung** cho cả tiến trình. Hai request liên tiếp cách nhau ít nhất `rate_limit_sec` giây (mặc định 3,1 giây). Code chỉ chờ phần còn thiếu, không chờ trọn.
- **Thử lại** khi gặp 429, lỗi 5xx hoặc lỗi mạng, tối đa tới số lần gửi ghi trong bảng dưới (lần đầu cộng các lần thử lại).
- **Backoff** (chờ tăng dần): lần chờ đầu bằng giá trị trong bảng, các lần sau gấp đôi, cộng một khoảng ngẫu nhiên nhỏ gọi là **jitter** để nhiều tiến trình không thử lại cùng lúc. Mỗi lần chờ có trần. Nếu S2 gửi header `Retry-After` (số giây phải chờ) thì code nghe theo header đó.

Hết lượt thử, code báo lỗi cho bài đó, không trả danh sách rỗng. Nhờ vậy bài lỗi không bị nhầm thành bài không có trích dẫn.

Bộ đếm request chỉ được ghi trong một tiến trình, nên khi chạy lại bằng `--resume` thì số đếm chỉ tính lần chạy cuối.

In [ ]:
print(
    f"Tham số đang dùng trong code: tối đa {rs._MAX_ATTEMPTS} lần gửi, backoff lần đầu {rs._BACKOFF_BASE_SEC:g} giây, "
    f"trần mỗi lần chờ {rs._MAX_WAIT_SEC:g} giây, khoảng cách mặc định giữa hai request "
    f"{inspect.signature(rs.scrape_references_for_paper).parameters['rate_limit_sec'].default:g} giây."
)

rows = []
for member_id, s in shards.items():
    counter = s["stats"].get("reference_scraper")
    phase = s["stats"].get("phases", {}).get("references")
    if s["stats"].get("resumed"):
        print(f"{member_id}: lần chạy cuối là --resume; số đếm chỉ tính phần chạy đó.")
    if counter is None or phase is None:
        print(f"{member_id}: chưa có số đếm request trong stats.json "
              "(cần Triết ghi reference_scraper và phases.references).")
        continue
    total_requests = counter["requests"]
    n_429 = counter["by_status"].get("429", 0)
    n_papers = phase["calls"]
    wait_sec = counter["backoff_sec"] + counter["rate_limit_wait_sec"]
    elapsed = phase["elapsed_sec"]
    rows.append({
        "MSSV": member_id,
        "số bài thử": n_papers,
        "request": total_requests,
        "request/bài": total_requests / n_papers if n_papers else np.nan,
        "mã 200": counter["by_status"].get("200", 0),
        "mã 404": counter["by_status"].get("404", 0),
        "mã 429": n_429,
        "lỗi mạng": counter["by_status"].get("network", 0),
        "lần thử lại": counter["retries"],
        "giây chờ backoff": counter["backoff_sec"],
        "giây chờ giới hạn tốc độ": counter["rate_limit_wait_sec"],
        "giây/bài (pha references)": elapsed / n_papers if n_papers else np.nan,
        "chờ/thời gian pha (%)": 100 * wait_sec / elapsed if elapsed else np.nan,
    })
    if total_requests and elapsed:
        print(f"{member_id}: {n_429}/{total_requests} request bị 429 ({n_429 / total_requests:.1%}); "
              f"{wait_sec / elapsed:.1%} thời gian pha references là chờ.")

if rows:
    display(pd.DataFrame(rows).set_index("MSSV").round(2))
else:
    print("Chưa có số đếm request.")

## 3. Dự phòng bằng .bib và .bbl

Khi S2 trả **404** (S2 không biết bài này), code đọc tệp `.bib` trong thư mục nguồn đã giải nén. Code chỉ giữ mục nào có khóa xuất hiện trong một lệnh trích dẫn của `.tex` (`\cite`, `\citep`, `\nocite`...), vì mục không được trích thì không phải trích dẫn của bài. Nếu không còn `.bib` dùng được thì code đọc `.bbl`. Đề chỉ giữ `.tex` và `.bib` sau khi giải nén, nên `.bbl` có thể đã bị xóa.

Dự phòng **không** dùng cho lỗi tạm thời (429, 5xx, mạng). Lỗi đó được thử lại; hết lượt thì báo lỗi. Nếu đổi sang `.bib` khi gặp lỗi mạng, số liệu sẽ bị che và sai.

Lưu ý khi đọc kết quả: `.bib` vừa là nguồn của các trích dẫn dự phòng, vừa là nguồn của câu hỏi BibTeX mà M3 ghép. Vì vậy các bài dùng dự phòng có thể dễ ghép hơn bài thường. Đề chưa quyết việc này, nhóm cần chốt trước khi báo cáo kết quả M3.

In [ ]:
if HAS_PAPERS:
    n = len(papers)
    via_bib = papers["via"].eq("bib_fallback")
    # Bài S2 trả 404: hoặc được cứu bằng .bib, hoặc không có .bib nào dùng được.
    s2_404 = papers["status"].eq("not_found") | via_bib
    print(f"Bài dùng dự phòng .bib: {via_bib.sum()}/{n} = {via_bib.mean():.1%}")
    if s2_404.sum():
        print(f"Bài S2 trả 404: {s2_404.sum()}; được cứu bằng .bib: "
              f"{via_bib.sum()}/{s2_404.sum()} = {via_bib.sum() / s2_404.sum():.1%}")
    else:
        print("Không có bài nào S2 trả 404.")
else:
    print("Chưa có reference_status: chưa tính được tỷ lệ dự phòng theo bài.")

QUALITY_ORDER = ["có ID arXiv", "chỉ có DOI", "chỉ có raw_citation", "không có gì dùng được"]


def quality_of(row):
    """Mức chất lượng của một trích dẫn lấy từ .bib: thông tin nào còn dùng được để ghép."""
    if pd.notna(row["ref_arxiv_id"]):
        return "có ID arXiv"
    if pd.notna(row["ref_doi"]):
        return "chỉ có DOI"
    if pd.notna(row["raw_citation"]):
        return "chỉ có raw_citation"
    return "không có gì dùng được"


if HAS_REFS:
    bib_refs = refs[refs["source"].eq("bib_fallback")]
    if len(bib_refs):
        quality = bib_refs.apply(quality_of, axis=1).value_counts().reindex(QUALITY_ORDER, fill_value=0)
        display(pd.DataFrame({"số trích dẫn từ .bib": quality}))
        print(f"Trong {len(bib_refs)} trích dẫn từ .bib: {quality['có ID arXiv']} có ID arXiv, "
              f"{quality['chỉ có DOI']} chỉ có DOI, {quality['chỉ có raw_citation']} chỉ có chuỗi gốc.")
    else:
        print("Không có trích dẫn nào lấy từ .bib.")
else:
    print("Chưa có references.parquet.")

## 4. Độ phủ và phân bố trích dẫn

M3 ghép mỗi mục BibTeX với các trích dẫn của cùng bài, dựa vào tiêu đề, tác giả và năm. Trích dẫn thiếu một trong các trường này vẫn được lưu, nhưng ghép khó đúng hơn. Mục này đo tỷ lệ có giá trị của từng cột, rồi kiểm tra dữ liệu có sạch không: cạnh trùng và năm bất thường.

Các chỉ số phân bố: **trung bình** là tổng chia cho số bài; **trung vị** là giá trị đứng giữa khi xếp các bài theo số trích dẫn, ít bị kéo lên bởi vài bài có rất nhiều trích dẫn; **P90** là mức mà 90% số bài có số trích dẫn không vượt quá.

Quy tắc dữ liệu: mỗi cặp (bài nguồn, tài liệu được trích) chỉ có một dòng. Trích dẫn có năm lớn hơn 2024 là đáng nghi (bài thuộc tháng 7/2024, nhưng bản sửa sau này có thể trích tài liệu mới hơn), nên chỉ được đếm và báo, không bị sửa.

In [ ]:
if not HAS_PAPERS:
    print("Chưa có reference_status: chưa tính được số trích dẫn theo bài.")
else:
    ok_papers = papers[papers["status"].eq("ok")]
    refs_per_paper = ok_papers["n_refs"].to_numpy()
    if len(refs_per_paper) == 0:
        print("Không có bài nào lấy không lỗi.")
    else:
        # dtype=object giữ số nguyên là số nguyên (không biến thành 4.00); số thực làm tròn 2 chữ số.
        summary = pd.Series({
            "số bài (lấy không lỗi)": int(len(refs_per_paper)),
            "trung bình": round(float(refs_per_paper.mean()), 2),
            "trung vị": round(float(np.median(refs_per_paper)), 2),
            "P90": round(float(np.percentile(refs_per_paper, 90)), 2),
            "lớn nhất": int(refs_per_paper.max()),
            "tổng số trích dẫn": int(refs_per_paper.sum()),
        }, name="trích dẫn mỗi bài", dtype=object)
        display(summary.to_frame())

        top = int(refs_per_paper.max())
        fig, ax = plt.subplots(figsize=(7, 3.8))
        # Mỗi cột ứng với một số nguyên: biên của cột nằm ở các điểm nửa nguyên.
        ax.hist(refs_per_paper, bins=np.arange(top + 2) - 0.5, color=BLUE, rwidth=0.7)
        ax.set_xticks(range(top + 1))
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))  # số bài là số nguyên
        ax.set_xlabel("Số trích dẫn của bài (mọi trích dẫn, kể cả không có arXiv ID)")
        ax.set_ylabel("Số bài")
        ax.set_title(f"{TAG}Hình 2. Số trích dẫn mỗi bài (n = {len(refs_per_paper)} bài lấy không lỗi)")
        ax.grid(axis="y")
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        plt.tight_layout()
        plt.show()

        print(f"Trung bình {refs_per_paper.mean():.2f} trích dẫn mỗi bài, trung vị {np.median(refs_per_paper):g}; "
              f"bài nhiều nhất có {top} trích dẫn.")

In [ ]:
if not HAS_REFS:
    print("Chưa có references.parquet.")
else:
    def has_value(cell):
        """True nếu ô có giá trị thật: danh sách không rỗng, hoặc giá trị không phải None/NaN."""
        if isinstance(cell, (list, np.ndarray)):
            return len(cell) > 0
        return pd.notna(cell)

    filled = {column: refs[column].apply(has_value).sum() for column in rs.REFERENCE_KEYS}
    completeness = pd.DataFrame({"có giá trị": pd.Series(filled)})
    completeness["tỷ lệ (%)"] = (100 * completeness["có giá trị"] / len(refs)).round(1)
    display(completeness)

    dup_arxiv = refs[refs["ref_arxiv_id"].notna()].duplicated(subset=["arxiv_id", "ref_arxiv_id"]).sum()
    dup_s2 = refs[refs["ref_s2_id"].notna()].duplicated(subset=["arxiv_id", "ref_s2_id"]).sum()
    print(f"Cạnh trùng theo (bài, ref_arxiv_id): {dup_arxiv}. Theo (bài, ref_s2_id): {dup_s2}.")
    print(f"Tổng: {len(refs)} trích dẫn từ {refs['arxiv_id'].nunique()} bài nguồn.")

In [ ]:
if not HAS_REFS:
    print("Chưa có references.parquet.")
else:
    years = refs["ref_year"].dropna().astype(int)
    missing_year = int(refs["ref_year"].isna().sum())
    print(f"Thiếu năm: {missing_year}/{len(refs)} = {missing_year / len(refs):.1%}")
    print(f"Năm lớn hơn 2024 (đáng nghi, chỉ đếm): {(years > 2024).sum()}")
    if len(years) == 0:
        print("Không có năm nào để vẽ.")
    else:
        by_year = years.value_counts().sort_index()
        display(by_year.rename("số trích dẫn").to_frame())

        fig, ax = plt.subplots(figsize=(7, 3.6))
        positions = np.arange(len(by_year))
        ax.bar(positions, by_year.values, color=BLUE, width=0.6)
        ax.set_xticks(positions)
        ax.set_xticklabels([str(year) for year in by_year.index])
        for x, value in zip(positions, by_year.values):
            ax.text(x, value + 0.05, str(value), ha="center", va="bottom", color=INK_SECONDARY)
        ax.set_ylim(0, by_year.max() * 1.25)
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))  # số trích dẫn là số nguyên
        ax.set_xlabel("Năm xuất bản của trích dẫn (theo S2)")
        ax.set_ylabel("Số trích dẫn")
        ax.set_title(f"{TAG}Hình 3. Trích dẫn theo năm (n = {len(years)} có năm; thiếu năm: {missing_year})")
        ax.grid(axis="y")
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        plt.tight_layout()
        plt.show()
        print(f"Năm có nhiều trích dẫn nhất: {by_year.idxmax()} ({by_year.max()} trích dẫn).")

## 5. Kết quả, nhận xét và giới hạn

Bảng dưới gom các chỉ số chính của từng thành viên có dữ liệu. Ký hiệu (a), (b), (c) như ở mục 1. Dòng ghi MẪU là dữ liệu giả, không dùng làm kết quả.

In [ ]:
if not shards:
    print("Chưa có shard nào.")
else:
    rows = []
    for member_id, s in shards.items():
        papers_m = papers[papers["member"].eq(member_id)]
        refs_m = refs[refs["member"].eq(member_id)]
        row = {
            "MSSV": member_id,
            "họ tên": MEMBERS[member_id],
            "dữ liệu": "MẪU" if s["is_mock"] else "thật",
            "số trích dẫn": len(refs_m),
        }
        if len(papers_m):
            ok_m = papers_m[papers_m["status"].eq("ok")]
            row.update({
                "N bài": len(papers_m),
                "(a) không lỗi (%)": 100 * papers_m["status"].eq("ok").mean(),
                "(b) có trích dẫn (%)": 100 * papers_m["n_refs"].gt(0).mean(),
                "(c) có arXiv ID (%)": 100 * papers_m["n_refs_arxiv"].gt(0).mean(),
                "bài dùng .bib": int(papers_m["via"].eq("bib_fallback").sum()),
                "trích dẫn TB/bài": ok_m["n_refs"].mean() if len(ok_m) else np.nan,
            })
        counter = s["stats"].get("reference_scraper")
        if counter and counter["requests"]:
            row["request"] = counter["requests"]
            row["mã 429 (%)"] = 100 * counter["by_status"].get("429", 0) / counter["requests"]
        rows.append(row)
    display(pd.DataFrame(rows).set_index("MSSV").round(2))

    # Kiểm tra: số bài trong reference_status phải khớp số bài trong publications.parquet.
    for member_id in shards:
        pub_path = SHARDS_DIR / member_id / "publications.parquet"
        if not pub_path.exists():
            print(f"{member_id}: không có publications.parquet, bỏ qua kiểm tra số bài.")
        elif HAS_PAPERS:
            n_pub = len(pd.read_parquet(pub_path))
            n_status = int(papers["member"].eq(member_id).sum())
            verdict = "khớp" if n_pub == n_status else "KHÔNG KHỚP, kiểm tra trước khi dùng số"
            print(f"{member_id}: publications.parquet {n_pub} bài, reference_status {n_status} bài: {verdict}")

**Nhận xét của nhóm:** *chưa có dữ liệu thật. Viết sau khi chạy trên shard thật, dựa vào các bảng và hình ở trên. Gợi ý các câu hỏi cần trả lời:*

- Các tỷ lệ (a), (b), (c) là bao nhiêu? Lỗi nào chiếm nhiều nhất?
- Dự phòng .bib cứu được bao nhiêu bài? Trích dẫn lấy từ .bib có đủ ID không?
- Chờ giới hạn tốc độ chiếm bao nhiêu thời gian của pha references?
- Trích dẫn có đủ tiêu đề, tác giả và năm để M3 dùng không?

**Giới hạn đã biết** (ghi nhận, không phải kết quả):

- S2 có thể không phân biệt các phiên bản của một bài [đề M1]. Trích dẫn là của bài, không theo từng phiên bản.
- Bài mới đăng có thể chậm được S2 lập chỉ mục, thường trong 1–7 ngày [đề M1].
- Số đếm request chỉ tính lần chạy cuối khi dùng `--resume`.
- Nhóm chưa quyết việc giữ hay bỏ `.bbl` (đề chỉ giữ `.tex` và `.bib`).

**Tài liệu tham khảo:**

- Đề M1, mục thu thập: https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/
- Semantic Scholar Graph API: https://api.semanticscholar.org/api-docs/graph
- AWS, *Exponential Backoff And Jitter*: https://aws.amazon.com/blogs/architecture/exponential-backoff-and-jitter/
- RFC 9110, mục 10.2.3 (Retry-After): https://www.rfc-editor.org/rfc/rfc9110#section-10.2.3